In [ ]:
import tensorflow as tf
# Allocate GPU memory only as needed. By default TensorFlow grabs all of it, which makes a second
# notebook (or kernel) fail with an out-of-memory error on the 4 GB laptop GPU.
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)


In [ ]:
import time
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from tensorflow import keras

# Load dataset
X, y = load_digits(return_X_y=True)
X = X / 16.0

# Split dataset
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Without early stopping
model1 = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(64,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model1.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

start = time.time()
h1 = model1.fit(X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=100, verbose=0)
time1 = time.time() - start

# With early stopping
model2 = keras.Sequential([
    keras.layers.Dense(128, activation='relu', input_shape=(64,)),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dense(10, activation='softmax')
])

model2.compile(optimizer='adam',
               loss='sparse_categorical_crossentropy',
               metrics=['accuracy'])

stop = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5,
    restore_best_weights=True
)

start = time.time()
h2 = model2.fit(X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=100, callbacks=[stop], verbose=0)
time2 = time.time() - start

# Compare results
print('Training time:', time1, 'vs', time2)
print('Epochs:', len(h1.history['loss']), 'vs', len(h2.history['loss']))
print('Val accuracy:', h1.history['val_accuracy'][-1],
      'vs', h2.history['val_accuracy'][-1])

# Plot validation loss
plt.plot(h1.history['val_loss'], label='Without Early Stopping')
plt.plot(h2.history['val_loss'], label='With Early Stopping')
plt.title('Validation Loss Comparison')
plt.xlabel('Epochs')
plt.ylabel('Validation Loss')
plt.legend()
plt.show()